Prerequisites & Installation

In [1]:
pip install transformers datasets evaluate accelerate gradio scikit-learn torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


Fine-Tuning and Evaluation

In [6]:
import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

def main():
    # 1. Load the AG News dataset (Using modern verified explicit repository)
    print("📥 Loading AG News dataset...")
    dataset = load_dataset("fancyzhx/ag_news")

    # Optional: Slice the dataset for faster prototyping/debugging
    # dataset["train"] = dataset["train"].shuffle(seed=42).select(range(10000))
    # dataset["test"] = dataset["test"].shuffle(seed=42).select(range(2000))

    id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
    label2id = {"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3}

    # 2. Tokenize and Preprocess
    print("🔤 Initializing tokenizer and preprocessing text...")
    model_checkpoint = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    def preprocess_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512)

    tokenized_datasets = dataset.map(preprocess_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # 3. Load Model
    print("🤖 Initializing bert-base-uncased model...")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=4,
        id2label=id2label,
        label2id=label2id
    )

    # 4. Define Evaluation Metrics
    accuracy_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        preds = np.argmax(predictions, axis=1)

        acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
        f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]

        return {"accuracy": acc, "f1_score": f1}

    # 5. Training Arguments (Updated keyword: eval_strategy)
    print("⚙️ Setting up training hyperparameters...")
    training_args = TrainingArguments(
        output_dir="./bert-news-classifier",
        learning_rate=2e-5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_score",
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to="none"
    )

    # 6. Initialize Trainer (Updated keyword: processing_class)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # 7. Fine-tune and Save
    print("🚀 Starting fine-tuning loop...")
    trainer.train()

    print("💾 Saving the best model and tokenizer...")
    trainer.save_model("./best_news_classifier")
    tokenizer.save_pretrained("./best_news_classifier")
    print("✅ Done! Training completed successfully.")

if __name__ == "__main__":
    main()

📥 Loading AG News dataset...
🔤 Initializing tokenizer and preprocessing text...


Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

🤖 Initializing bert-base-uncased model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

💾 Saving the best model and tokenizer...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Done! Training completed successfully.


Deployment with Gradio


In [9]:
import gradio as gr
from transformers import pipeline

model_path = "./best_news_classifier"
print(f"🔮 Loading fine-tuned pipeline from {model_path}...")

# 1. Clean pipeline initialization (removes the nesting bug)
classifier = pipeline("text-classification", model=model_path, tokenizer=model_path)

def classify_news(headline):
    if not headline.strip():
        return {}

    # 2. Request all category scores during the live call
    results = classifier(headline, top_k=None)

    # If the output is wrapped in an extra list, extract the inner one safely
    if isinstance(results[0], list):
        results = results[0]

    # 3. Build a clean dictionary for Gradio's Label component
    predictions = {item["label"]: float(item["score"]) for item in results}
    return predictions

# 4. Build Layout
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📰 News Topic Classifier (BERT)")
    gr.Markdown("Type a headline to view the model's full probability distribution across all 4 topics.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="News Headline / Article Intro",
                placeholder="Type or paste a headline here...",
                lines=4
            )
            submit_btn = gr.Button("Classify Headline", variant="primary")

        with gr.Column():
            # Grabs predictions and maps them to clean interactive bar meters
            output_chart = gr.Label(label="Topic Probabilities", num_top_classes=4)

    gr.Examples(
        examples=[
            ["Nike signs a record-breaking $500 million sponsorship contract with an Olympic gold medalist, driving their stock up by 3%."],
            ["Cyberwarfare units backed by foreign states successfully bypassed a government database firewall last night."]
        ],
        inputs=input_text
    )

    submit_btn.click(fn=classify_news, inputs=input_text, outputs=output_chart)

print("🚀 Starting Visual Web Interface...")
demo.launch(share=True)

🔮 Loading fine-tuned pipeline from ./best_news_classifier...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_600/1039642331.py:26: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


🚀 Starting Visual Web Interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a83dfb10e114537239.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
